# MeanFlow — reproduce both results

One notebook, both tiers of the assignment:

1. **Required: 3-image overfit** (~10 min on an L4) — train on 3 fixed Imagenette crops and reproduce them with one-step generation, verified on held-out noise.
2. **Bonus: 10-class Imagenette** (~1.5 h on an L4) — train unconditionally on all 9,469 Imagenette training images; one-step samples show recognizable structure.

**Before running:** Runtime → Change runtime type → GPU.

Each section is self-contained after the setup cell. For long-run conveniences (periodic Drive saves, disconnect recovery via `--resume-from`), see `Run_Imagenette10_in_Colab.ipynb`.

In [ ]:
!git clone https://github.com/nonplayercharacter3/MeanFlow-one-step-pixel-generation.git meanflow
%cd meanflow
!nvidia-smi -L

## 1. Required tier: 3-image overfit

The final documented recipe (report section 3.2). Trains in ~10 minutes and reaches mean nearest-image `sample_mse ≈ 0.009` — one-step samples visually indistinguishable from the three training images.

Evaluation is assignment-free: MeanFlow's learned flow picks its own noise→image basins, so each sample is scored against its *nearest* training image, and each image by its best reproduction across the fixed eval noises.

In [ ]:
!python train.py \
  --images data/overfit3/image_0.png data/overfit3/image_1.png data/overfit3/image_2.png \
  --steps 2500 --batch-size 64 --lr 5e-4 --hidden-channels 128 --num-blocks 2 \
  --time-sampling logit_normal --equal-time-probability 0.5 --endpoint-probability 0.1 \
  --loss-weight-power 0.75 --adaptive-lr --lr-patience 150 --ema-decay 0.99 --reweight-images \
  --sample-every 500 --checkpoint-every 1000 --output-dir outputs/overfit3_final

### Held-out check

16 fresh noises from a seed never used in training: every sample should land cleanly on *some* training image, and every image should be covered by at least one noise.

In [ ]:
!python sample.py \
  --checkpoint outputs/overfit3_final/checkpoint_best.pt \
  --images data/overfit3/image_0.png data/overfit3/image_1.png data/overfit3/image_2.png \
  --num-samples 16

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

out = Path("outputs/overfit3_final")
rows = [
    ("clean_grid.png", "the 3 training images"),
    ("sample_best_grid.png", "one-step samples from the 8 fixed eval noises (sorted by nearest image)"),
    ("samples_many_noises.png", "held-out: one-step samples from 16 fresh noises (sample.py)"),
]
fig, axes = plt.subplots(len(rows), 1, figsize=(14, 6))
for axis, (name, title) in zip(axes, rows):
    axis.imshow(Image.open(out / name))
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()
plt.show()

Image.open(out / "loss_curve.png")

## 2. Bonus tier: 10-class Imagenette

Download the dataset (160px edition, ~100 MB), then train unconditionally on the full train split. This is the exact configuration behind the report's section 3.3 figures.

Note: the training loss is expected to hover just below 1.0 — with `--loss-weight-power 1.0` each sample contributes `err/(err + 1e-3)`, which compresses the scale. Judge progress by the sample grids, not the loss.

In [ ]:
!wget -q https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz -O data/imagenette2-160.tgz
!tar -xzf data/imagenette2-160.tgz -C data/
!find data/imagenette2-160/train -name '*.JPEG' | wc -l

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train.py \
  --data-dir data/imagenette2-160/train \
  --steps 20000 --batch-size 64 --lr 3e-4 --lr-decay-steps 20000 \
  --time-sampling logit_normal --equal-time-probability 0.5 --endpoint-probability 0.1 \
  --loss-weight-power 1.0 \
  --eval-every 100 --sample-every 1000 --checkpoint-every 2000 \
  --num-eval-noises 16 --num-workers 4 --output-dir outputs/imagenette10

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

out = Path("outputs/imagenette10")
# Numeric sort: past step 9999 alphabetical order would misreport the latest grid.
grids = sorted(out.glob("sample_step_*.png"), key=lambda p: int(p.stem.split("_")[-1]))

rows = [(out / "train_batch_grid.png", "training data (reference)")]
for step_grid in (grids[0], grids[len(grids) // 4], grids[-1]):
    rows.append((step_grid, f"one-step samples ({step_grid.name})"))

fig, axes = plt.subplots(len(rows), 1, figsize=(14, 8))
for axis, (path, title) in zip(axes, rows):
    axis.imshow(Image.open(path))
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()
plt.show()

history = pd.read_csv(out / "loss_history.csv")
plt.figure(figsize=(7, 4))
plt.semilogy(history["step"], history["loss"], alpha=0.5, label="loss")
plt.semilogy(history["step"], history["smoothed_loss"], "k", linewidth=2, label="smoothed_loss")
plt.xlabel("step")
plt.legend()
plt.title("10-class run: training curves")
plt.show()

## Save everything to Google Drive (optional)

Colab VMs are ephemeral; run this to keep the checkpoints and grids.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

import shutil
from pathlib import Path

drive_dir = Path('/content/drive/MyDrive/MeanFlow_outputs')
drive_dir.mkdir(parents=True, exist_ok=True)
shutil.copytree('outputs', drive_dir, dirs_exist_ok=True)
print(f"Copied outputs/ to {drive_dir}")